In [1]:
!pip install xlrd
import pandas as pd
import unicodedata

Importation des données de toutes les élections

In [2]:
all_elections = pd.read_csv("/home/onyxia/work/projet3A/donnees_electorales/candidats_results.txt", sep=";")
#mini modif manuelle à cause d'une erreur
all_elections.loc[22340396, 'Binôme'] = all_elections.loc[22160435, 'Binôme']

/tmp/ipykernel_11136/3451900702.py:1: DtypeWarning: Columns (2,4,6,7,8,12,13,14,15,16,17) have mixed types. Specify dtype option on import or set low_memory=False.
  all_elections = pd.read_csv("/home/onyxia/work/projet3A/donnees_electorales/candidats_results.txt", sep=";")


In [3]:
muni_2014 = pd.read_csv("/home/onyxia/work/projet3A/donnees_electorales/MN14_Bvot_T1T2.txt", sep=";",
    encoding="latin-1", skiprows=17, header= None)
muni_2014.columns = ['id_tour', 'Code du département', 'Code de la commune', 'Nom de la commune',
'Code du b.vote', 'Inscrits', 'Votants', 'Exprimes', 'num liste', 'Prénom', 'Nom', 'Nuance', 'Voix']
muni_2014['id_tour'] = "t" + muni_2014['id_tour'].astype(str)

/tmp/ipykernel_11136/1504886825.py:1: DtypeWarning: Columns (1,4) have mixed types. Specify dtype option on import or set low_memory=False.
  muni_2014 = pd.read_csv("/home/onyxia/work/projet3A/donnees_electorales/MN14_Bvot_T1T2.txt", sep=";",


In [4]:
def simplifier_prenom(text):
    if pd.isna(text):
        return None   # ou "" ou text
    # Normalise en forme NFD (décompose les accents)
    text_normalized = unicodedata.normalize('NFD', text)
    # Garde seulement les caractères ASCII (supprime les accents)
    text_ascii = text_normalized.encode('ascii', 'ignore').decode('utf-8')
    text_clean = text_ascii.replace('-', ' ')
    return text_clean

## Etape initiale : constituer une liste des nuances de tous les candidats

In [5]:
all_elections.head()

,id_election,id_brut_miom,Code du département,Code de la commune,Code du b.vote,N°Panneau,Libellé Abrégé Liste,Libellé Etendu Liste,Nom Tête de Liste,Voix,% Voix/Ins,% Voix/Exp,Sexe,Nom,Prénom,Nuance,Binôme,Liste
0,2019_euro_t1,01001_0001,01,1,0001,1.0,La France Insoumise,La France Insoumise,AUBRY Manon,13.0,2.16,4.14,NaN,NaN,NaN,NaN,NaN,NaN
1,2019_euro_t1,01002_0001,01,2,0001,1.0,La France Insoumise,La France Insoumise,AUBRY Manon,6.0,2.86,4.44,NaN,NaN,NaN,NaN,NaN,NaN
2,2019_euro_t1,01004_0001,01,4,0001,1.0,La France Insoumise,La France Insoumise,AUBRY Manon,39.0,3.71,8.23,NaN,NaN,NaN,NaN,NaN,NaN
3,2019_euro_t1,01004_0002,01,4,0002,1.0,La France Insoumise,La France Insoumise,AUBRY Manon,42.0,3.80,7.78,NaN,NaN,NaN,NaN,NaN,NaN
4,2019_euro_t1,01004_0003,01,4,0003,1.0,La France Insoumise,La France Insoumise,AUBRY Manon,31.0,2.93,5.63,NaN,NaN,NaN,NaN,NaN,NaN


##### Traitement des données des candidats aux départementales, qui ont un format "binome"

In [6]:
elec_binome = all_elections[all_elections["Binôme"].notna()]
liste_elec_binome = elec_binome['id_election'].unique()
election_dep = all_elections[all_elections['id_election'].isin(liste_elec_binome)]
election_dep.drop_duplicates(subset=["id_election", "id_brut_miom", "Code du département","Nuance", "Binôme"], inplace=True)
election_dep = election_dep[["id_election", "id_brut_miom", "Code du département","Nuance", "Binôme"]]
election_dep[["candidat1", "candidat2"]] = election_dep["Binôme"].str.split(" et ", expand=True)

# Enlever les titres "M" ou "Mme" au début
election_dep["candidat1"] = election_dep["candidat1"].str.replace(r"^(M\.|Mme)\s+", "", regex=True)
election_dep["candidat2"] = election_dep["candidat2"].str.replace(r"^(M\.|Mme)\s+", "", regex=True)

election_dep[[ "nom1", "prenom1"]] = election_dep["candidat1"].str.split(" ", n=1, expand=True)
election_dep["prenom1"] = election_dep["prenom1"].str.lower()
election_dep[[ "nom2","prenom2"]] = election_dep["candidat2"].str.split(" ", n=1, expand=True)
election_dep["prenom2"] = election_dep["prenom2"].str.lower()

election_dep_long = pd.wide_to_long(election_dep, 
                          stubnames=["prenom", "nom"], 
                          i=["id_election", "id_brut_miom", "Code du département","Nuance", "Binôme"], 
                          j="num_binôme",   # nouveau suffixe qui indiquera 1 ou 2
                          sep="")    # pas de séparateur entre stubname et numéro

# Réinitialiser l'index pour obtenir un DataFrame classique
election_dep_long = election_dep_long.reset_index()
election_dep_long['Nuance'] = election_dep_long['Nuance'].str.replace(r'^BC-', '', regex=True)
election_dep_long.drop(['Binôme', 'num_binôme', 'candidat1', 'candidat2'], axis = 1, inplace= True)

/tmp/ipykernel_11136/678289474.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  election_dep.drop_duplicates(subset=["id_election", "id_brut_miom", "Code du département","Nuance", "Binôme"], inplace=True)


In [7]:
election_dep_long.head()

,id_election,id_brut_miom,Code du département,Nuance,prenom,nom
0,2021_dpmt_t2,01001_0001,01,UCD,patricia,CHMARA
1,2021_dpmt_t2,01001_0001,01,UCD,patrick,MATHIAS
2,2021_dpmt_t2,01002_0001,01,UG,fabrice,PEREYRON
3,2021_dpmt_t2,01002_0001,01,UG,marie-céline,RAY
4,2021_dpmt_t2,01004_0001,01,UG,fabrice,PEREYRON


##### Traitement des candidats hors municipales et départementales

In [8]:
election_hors_dep = all_elections[~all_elections['id_election'].isin(liste_elec_binome)]
election_hors_dep_et_muni = election_hors_dep[election_hors_dep["id_election"].str.contains("muni", case=False, na=False)]
election_hors_dep_et_muni = election_hors_dep_et_muni[['id_election', 'id_brut_miom', 'Code du département', 'Nom', 'Prénom', 'Nuance']]

In [9]:
election_hors_dep_et_muni.head()

,id_election,id_brut_miom,Code du département,Nom,Prénom,Nuance
4267946,2020_muni_t2,01012_0001,01,PIRES,Hervé,NC
4267947,2020_muni_t2,01034_0001,01,FOGNINI,Jean-Marc,LDIV
4267948,2020_muni_t2,01034_0002,01,FOGNINI,Jean-Marc,LDIV
4267949,2020_muni_t2,01034_0003,01,FOGNINI,Jean-Marc,LDIV
4267950,2020_muni_t2,01034_0004,01,FOGNINI,Jean-Marc,LDIV


In [10]:
election_dep_long.rename(columns={'prenom': 'Prénom'}, inplace=True)
election_dep_long.rename(columns={'nom': 'Nom'}, inplace=True)
election_dep_long.rename(columns={'Code du département': 'dep'}, inplace=True)
election_hors_dep_et_muni.rename(columns={'Code du département': 'dep'}, inplace=True)
liste_candidats_nuance_hors_muni = pd.concat([election_dep_long, election_hors_dep_et_muni], ignore_index=True)
liste_candidats_nuance_hors_muni['annee'] = liste_candidats_nuance_hors_muni['id_election'].str[:4]
liste_candidats_nuance_hors_muni.head()

,id_election,id_brut_miom,dep,Nuance,Prénom,Nom,annee
0,2021_dpmt_t2,01001_0001,01,UCD,patricia,CHMARA,2021
1,2021_dpmt_t2,01001_0001,01,UCD,patrick,MATHIAS,2021
2,2021_dpmt_t2,01002_0001,01,UG,fabrice,PEREYRON,2021
3,2021_dpmt_t2,01002_0001,01,UG,marie-céline,RAY,2021
4,2021_dpmt_t2,01004_0001,01,UG,fabrice,PEREYRON,2021


### On s'intéresse au score des municipales

In [11]:
muni_2014['id_brut_miom'] = muni_2014['Code du département'].astype(str) + muni_2014['Code de la commune'].astype(str) + "_" + muni_2014['Code du b.vote'].astype(str)
muni_2014['id_election'] = "2014_muni_" + muni_2014['id_tour'].astype(str)
muni_2014['ident_election_ville'] = muni_2014["id_brut_miom"].str[:-5]
#muni_2014.drop('id_tour', inplace= True)

In [12]:
donnees_restreintes_muni = all_elections[all_elections["id_election"].str.contains("muni", case=False, na=False)]
donnees_restreintes_muni['ident_election_ville'] = donnees_restreintes_muni["id_brut_miom"].str[:5]

donnees_muni = pd.concat([donnees_restreintes_muni,muni_2014])

/tmp/ipykernel_11136/3570909418.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  donnees_restreintes_muni['ident_election_ville'] = donnees_restreintes_muni["id_brut_miom"].str[:5]


In [13]:
# ---- Étape 1 : calcul des voix totales par élection et ville ----
voix_totales = (
    donnees_muni
    .groupby(["id_election", "ident_election_ville"], as_index=False)
    .agg(voix_total_ville_elec=("Voix", "sum"))
)

In [15]:
# ---- Étape 2 : calcul des voix par candidat ----
donnees_muni_long = (
    donnees_muni
    .groupby(["id_election", "ident_election_ville", "Nom", "Prénom", "Nuance"], as_index=False)
    .agg(voix_totales_candidat=("Voix", "sum"))
    .merge(voix_totales, on=["id_election", "ident_election_ville"], how="left")
    .assign(voix_pct=lambda d: d["voix_totales_candidat"] / d["voix_total_ville_elec"] * 100)
    .sort_values(["id_election", "ident_election_ville", "voix_pct"], ascending=[True, True, False])
)

# ---- Étape 3 : classement des deux premiers ----
donnees_muni_wide = (
    donnees_muni_long
    .assign(rang=lambda d: d.groupby(["id_election", "ident_election_ville"]).cumcount() + 1)
    .query("rang <= 2")
    .pivot(index=["id_election", "ident_election_ville"],
           columns="rang",
           values=["Prénom", "Nom", "Nuance","voix_pct"])
)

# Aplatir les noms de colonnes hiérarchiques comme "rang1_Prénom"
donnees_muni_wide.columns = [
    f"rang{r}_{v}" for v, r in donnees_muni_wide.columns
]
donnees_muni_wide = donnees_muni_wide.reset_index()

# ---- Étape 5 : extraire les variables election / tour ----
donnees_muni_wide = (
    donnees_muni_wide
    .assign(
        election=lambda d: d["id_election"].str[:9],
        tour=lambda d: d["id_election"].str[10:12])
    .groupby(["election", "ident_election_ville"], group_keys=False)
    .filter(lambda g: not ("t1" in g["tour"].values and "t2" in g["tour"].values and g["tour"].eq("t1").any()))
)


In [16]:
donnees_muni_wide.head()

,id_election,ident_election_ville,rang1_Prénom,rang2_Prénom,rang1_Nom,rang2_Nom,rang1_Nuance,rang2_Nuance,rang1_voix_pct,rang2_voix_pct,election,tour
1,2008_muni_t1,01014,Liliane,Michel,MAISSIAT,GAUTHIER,LMAJ,LDVD,78.73701,21.26299,2008_muni,t1
2,2008_muni_t1,01033,Régis,Guy,PETIT,LARMANJAT,LMAJ,LSOC,58.224883,41.775117,2008_muni,t1
5,2008_muni_t1,01053,Jean-François,Christophe,DEBAT,FEILLENS,LSOC,LMAJ,55.371775,34.802731,2008_muni,t1
7,2008_muni_t1,01142,Bernard,Bernard,SIMPLEX,LOBIETTI,LDVD,LMAJ,58.760951,41.239049,2008_muni,t1
8,2008_muni_t1,01143,Etienne,NaN,BLANC,NaN,LMAJ,NaN,100.0,NaN,2008_muni,t1


In [17]:
donnees_muni_long[donnees_muni_long['Nom'] == "VUILLEROD"]

,id_election,ident_election_ville,Nom,Prénom,Nuance,voix_totales_candidat,voix_total_ville_elec,voix_pct
16757,2014_muni_t1,01340,VUILLEROD,Janick,NC,77.0,975.0,7.897436
16758,2014_muni_t1,01340,VUILLEROD,René,NC,72.0,975.0,7.384615


In [18]:
donnees_muni_long.head()

,id_election,ident_election_ville,Nom,Prénom,Nuance,voix_totales_candidat,voix_total_ville_elec,voix_pct
1,2008_muni_t1,01004,EXPOSITO,Josiane,LUG,1341.0,4272.0,31.390449
0,2008_muni_t1,01004,CASTELLANO,Sandrine,LDVD,1327.0,4272.0,31.062734
2,2008_muni_t1,01004,PAVIER,Bernard,LGC,820.0,4272.0,19.194757
3,2008_muni_t1,01004,SASSO,Jean-Marc,LDVD,784.0,4272.0,18.352060
5,2008_muni_t1,01014,MAISSIAT,Liliane,LMAJ,985.0,1251.0,78.737010


In [19]:
#on prépare la suite en créant un fichier listant les têtes de liste des municipales et leur couleur politique
liste_candidats_nuance_muni = (
                    donnees_muni_long
                    .assign(annee = lambda d : d["id_election"].str[0:4])
                    .assign(dep = lambda d : d["ident_election_ville"].str[0:2])
                    .loc[:, ["id_election", "ident_election_ville","annee","dep", "Nom", "Prénom", "Nuance"]]
)
liste_candidats_nuance_muni.head()


,id_election,ident_election_ville,annee,dep,Nom,Prénom,Nuance
1,2008_muni_t1,01004,2008,01,EXPOSITO,Josiane,LUG
0,2008_muni_t1,01004,2008,01,CASTELLANO,Sandrine,LDVD
2,2008_muni_t1,01004,2008,01,PAVIER,Bernard,LGC
3,2008_muni_t1,01004,2008,01,SASSO,Jean-Marc,LDVD
5,2008_muni_t1,01014,2008,01,MAISSIAT,Liliane,LMAJ


#### fusion de différentes variables

In [71]:
liste_candidats_nuance = pd.concat([liste_candidats_nuance_hors_muni,liste_candidats_nuance_muni], ignore_index=True)
liste_candidats_nuance.head()
liste_candidats_nuance['prenom'] =liste_candidats_nuance['Prénom'].str.lower().apply(simplifier_prenom)
liste_candidats_nuance['nom'] =liste_candidats_nuance['Nom'].str.lower().apply(simplifier_prenom)
liste_candidats_nuance.dropna(subset=['prenom'], inplace=True)
liste_candidats_nuance.drop(columns=["id_election","id_brut_miom","ident_election_ville","Prénom", "Nom"], inplace=True)
liste_candidats_nuance.drop_duplicates(inplace=True)
print(liste_candidats_nuance_hors_muni.shape)
print(liste_candidats_nuance_muni.shape)
print(liste_candidats_nuance.shape)

(3067490, 7)
(1418975, 7)
(1685858, 5)


In [72]:
liste_candidats_nuance.head()

,dep,Nuance,annee,prenom,nom
0,01,UCD,2021,patricia,chmara
1,01,UCD,2021,patrick,mathias
2,01,UG,2021,fabrice,pereyron
3,01,UG,2021,marie celine,ray
20,01,RN,2021,clarisse,cathaud


Importation des données des intercommunalités

In [73]:
grp_2013 = pd.read_csv("/home/onyxia/work/projet3A/donnees_electorales/Liste_des_groupements_en_2013.csv", encoding="latin-1", sep = ";")
grp_2019 = pd.read_csv("/home/onyxia/work/projet3A/donnees_electorales/Liste_des_groupements_en_2019.csv", encoding="latin-1", sep = ";")
grp_2025 = pd.read_csv("/home/onyxia/work/projet3A/donnees_electorales/Liste_des_groupements_en_2024.csv", encoding="latin-1", sep = ";")
grp_2013['annee'] = "2008"
grp_2019['annee'] = "2014"
grp_2025['annee'] = "2020"

grp_2013['Nom Président'] = grp_2013['Nom Président'].str.lower().apply(simplifier_prenom)
grp_2019['Nom Président'] = grp_2019['Nom Président'].str.lower().apply(simplifier_prenom)
grp_2025['Nom Président'] = grp_2025['Nom Président'].str.lower().apply(simplifier_prenom)

grp_2013['prenom'] = grp_2013['Prénom Président'].str.lower().apply(simplifier_prenom)
grp_2019['prenom'] = grp_2019['Prénom Président'].str.lower().apply(simplifier_prenom)
grp_2025['prenom'] = grp_2025['Prénom Président'].str.lower().apply(simplifier_prenom)


grp_2013['dep'] = grp_2013["Département siège"].str[:2]
grp_2019['dep'] = grp_2019["Département siège"].str[:2]
grp_2025['dep'] = grp_2025["Département siège"].str[:2]



In [76]:
liste_candidats_nuance[(liste_candidats_nuance['nom'] == "levrat")]

,dep,Nuance,annee,prenom,nom
1664047,71,NC,2020,marie odile,levrat
1674591,58,NC,2020,michel,levrat
1830550,01,NC,2020,michel,levrat
2028733,39,NC,2020,bernard,levrat
2107871,31,NC,2020,dominique,levrat
2225000,01,NC,2020,valentin,levrat
2375532,1,LDVG,2014,gisele,levrat
2379215,1,NC,2014,didier,levrat
2379235,1,NC,2014,michel,levrat
2552503,31,NC,2014,dominique,levrat


In [77]:
grp_2019['annee'] = grp_2019['annee'].astype(int)
grp_2019['dep'] = grp_2019['dep'].astype(str)
grp_2019 = grp_2019.sort_values("annee")
grp_2019['Nom Président'] = grp_2019['Nom Président'].apply(simplifier_prenom)

liste_candidats_nuance['annee'] = liste_candidats_nuance['annee'].astype(int)
liste_candidats_nuance['dep'] = liste_candidats_nuance['dep'].astype(str)
liste_candidats_nuance = liste_candidats_nuance.sort_values("annee")

In [80]:
grp_2019_avec_nuance_test_proxi = pd.merge_asof(
    grp_2019,
    liste_candidats_nuance,
    on="annee",          # recherche de l'année la plus proche
    left_by=["prenom", "Nom Président", "dep"],
    right_by=["prenom", "nom", "dep"],
    direction="nearest"   # année la plus proche (avant ou après)
)

In [81]:
grp_2019_avec_nuance_test_proxi.head(50)

,Région siège,Département siège,Arrondissement siège,Commune siège,N° SIREN,Nom du groupement,Nature juridique,Syndicat à la carte,Groupement interdépartemental,Date de création,...,Nombre de compétences exercées,Mode de financement,Civilité Président,Prénom Président,Nom Président,annee,prenom,dep,Nuance,nom
0,84 - Auvergne-Rhône-Alpes,01 - Ain,4 - Nantua,210102836 - Oyonnax,200042935,Haut - Bugey Agglomération,CA,0,0,01/01/2014,...,35,FPU,M.,Jean,deguerry,2014,jean,01,LDVD,deguerry
1,84 - Auvergne-Rhône-Alpes,01 - Ain,2 - Bourg-en-Bresse,210100533 - Bourg-en-Bresse,200071751,CA du Bassin de Bourg-en-Bresse,CA,0,0,16/12/2016,...,41,FPU,M.,JEAN FRANCOIS,debat,2014,jean francois,01,LUG,debat
2,84 - Auvergne-Rhône-Alpes,01 - Ain,3 - Gex,210101739 - Gex,240100750,CA du Pays de Gex,CA,0,0,31/05/1995,...,35,FPU,M.,Christophe,bouvier,2014,christophe,01,LDVD,bouvier
3,84 - Auvergne-Rhône-Alpes,01 - Ain,4 - Nantua,210101994 - Jujurieux,200029999,CC Rives de l'Ain - Pays du Cerdon,CC,0,0,25/11/2011,...,18,FPU,M.,Thierry,dupuis,2014,thierry,01,LDVG,dupuis
4,84 - Auvergne-Rhône-Alpes,01 - Ain,1 - Belley,210100343 - Belley,200040350,CC Bugey Sud,CC,0,0,01/01/2014,...,22,FPU,M.,RENE,vuillerod,2014,rene,01,NC,vuillerod
5,84 - Auvergne-Rhône-Alpes,01 - Ain,2 - Bourg-en-Bresse,210104279 - Trévoux,200042497,CC Dombes Saône Vallée,CC,0,0,01/01/2014,...,28,FPU,M.,Bernard,grison,2014,bernard,01,LDVD,grison
6,84 - Auvergne-Rhône-Alpes,01 - Ain,2 - Bourg-en-Bresse,210100939 - Châtillon-sur-Chalaronne,200069193,CC de la Dombes,CC,0,0,01/12/2016,...,21,FPU,M.,MICHEL,girer,2014,michel,01,NaN,NaN
7,84 - Auvergne-Rhône-Alpes,01 - Ain,2 - Bourg-en-Bresse,210102588 - Montceaux,200070118,CC Val de Saône Centre,CC,0,0,06/12/2016,...,23,FPU,M.,Jean Claude,deschizeaux,2014,jean claude,01,LDVD,deschizeaux
8,84 - Auvergne-Rhône-Alpes,01 - Ain,2 - Bourg-en-Bresse,210103065 - Pont-de-Veyle,200070555,CC de la Veyle,CC,0,0,08/12/2016,...,22,FPU,M.,CHRISTOPHE,greffet,2014,christophe,01,NC,greffet
9,84 - Auvergne-Rhône-Alpes,01 - Ain,2 - Bourg-en-Bresse,210100269 - Bâgé-le-Châtel,200071371,CC Bresse et Saône,CC,0,0,15/12/2016,...,22,FPU,M.,GUY,billoudet,2014,guy,01,LDVD,billoudet


In [82]:
print(grp_2019_avec_nuance_test_proxi.shape)
print(grp_2019_avec_nuance_test_proxi['Nuance'].isna().sum())

(11226, 25)
3014


In [100]:
grp_2019_avec_nuance = pd.merge(
    grp_2019,
    liste_candidats_nuance,
    how="left",  
    left_on=["prenom", "Nom Président", "annee", "dep"],
    right_on=["prenom", "Nom", "annee", "dep"]
)
grp_2019_avec_nuance.head(40)

,Région siège,Département siège,Arrondissement siège,Commune siège,N° SIREN,Nom du groupement,Nature juridique,Syndicat à la carte,Groupement interdépartemental,Date de création,...,Nombre de compétences exercées,Mode de financement,Civilité Président,Prénom Président,Nom Président,annee,prenom,dep,Nuance,Nom
0,84 - Auvergne-Rhône-Alpes,01 - Ain,4 - Nantua,210102836 - Oyonnax,200042935,Haut - Bugey Agglomération,CA,0,0,01/01/2014,...,35,FPU,M.,Jean,DEGUERRY,2014,jean,01,NaN,NaN
1,84 - Auvergne-Rhône-Alpes,01 - Ain,2 - Bourg-en-Bresse,210100533 - Bourg-en-Bresse,200071751,CA du Bassin de Bourg-en-Bresse,CA,0,0,16/12/2016,...,41,FPU,M.,JEAN FRANCOIS,DEBAT,2014,jean francois,01,NaN,NaN
2,84 - Auvergne-Rhône-Alpes,01 - Ain,3 - Gex,210101739 - Gex,240100750,CA du Pays de Gex,CA,0,0,31/05/1995,...,35,FPU,M.,Christophe,BOUVIER,2014,christophe,01,NaN,NaN
3,84 - Auvergne-Rhône-Alpes,01 - Ain,4 - Nantua,210101994 - Jujurieux,200029999,CC Rives de l'Ain - Pays du Cerdon,CC,0,0,25/11/2011,...,18,FPU,M.,Thierry,DUPUIS,2014,thierry,01,NaN,NaN
4,84 - Auvergne-Rhône-Alpes,01 - Ain,1 - Belley,210100343 - Belley,200040350,CC Bugey Sud,CC,0,0,01/01/2014,...,22,FPU,M.,RENE,VUILLEROD,2014,rene,01,NaN,NaN
5,84 - Auvergne-Rhône-Alpes,01 - Ain,2 - Bourg-en-Bresse,210104279 - Trévoux,200042497,CC Dombes Saône Vallée,CC,0,0,01/01/2014,...,28,FPU,M.,Bernard,GRISON,2014,bernard,01,NaN,NaN
6,84 - Auvergne-Rhône-Alpes,01 - Ain,2 - Bourg-en-Bresse,210100939 - Châtillon-sur-Chalaronne,200069193,CC de la Dombes,CC,0,0,01/12/2016,...,21,FPU,M.,MICHEL,GIRER,2014,michel,01,NaN,NaN
7,84 - Auvergne-Rhône-Alpes,01 - Ain,2 - Bourg-en-Bresse,210102588 - Montceaux,200070118,CC Val de Saône Centre,CC,0,0,06/12/2016,...,23,FPU,M.,Jean Claude,DESCHIZEAUX,2014,jean claude,01,NaN,NaN
8,84 - Auvergne-Rhône-Alpes,01 - Ain,2 - Bourg-en-Bresse,210103065 - Pont-de-Veyle,200070555,CC de la Veyle,CC,0,0,08/12/2016,...,22,FPU,M.,CHRISTOPHE,GREFFET,2014,christophe,01,NaN,NaN
9,84 - Auvergne-Rhône-Alpes,01 - Ain,2 - Bourg-en-Bresse,210100269 - Bâgé-le-Châtel,200071371,CC Bresse et Saône,CC,0,0,15/12/2016,...,22,FPU,M.,GUY,BILLOUDET,2014,guy,01,NaN,NaN


In [ ]:
print(grp_2019.shape)
print(grp_2019_avec_nuance.shape)
print(grp_2019_avec_nuance['Nuance'].isna().sum())

(11226, 23)
(19621, 28)
5142


In [ ]:
non_matched = grp_2019_avec_nuance[grp_2019_avec_nuance['Nuance'].isna()].drop(columns=['Nuance'])
grp_2019_avec_nuance_max = non_matched.merge(liste_candidats_nuance, 
                            left_on=['prenom', 'Nom Président', 'dep'],
                            right_on=['prenom', 'Nom', 'dep'],
                            how="left", suffixes=('', '_2'))


In [168]:
grp_2019_avec_nuance_max.columns

Index(['Région siège', 'Département siège', 'Arrondissement siège',
       'Commune siège', 'N° SIREN', 'Nom du groupement', 'Nature juridique',
       'Syndicat à la carte', 'Groupement interdépartemental',
       'Date de création', 'Date d'effet', 'Mode de répartition des sièges',
       'Autre mode de répartition des sièges', 'Nombre de membres',
       'Population', 'Nombre de compétences exercées', 'Mode de financement',
       'Civilité Président', 'Prénom Président', 'Nom Président', 'annee',
       'prenom', 'dep', 'id_election', 'id_brut_miom', 'Prénom', 'Nom',
       'id_election_2', 'id_brut_miom_2', 'Nuance', 'Prénom_2', 'Nom_2',
       'annee_2'],
      dtype='object')

In [169]:
print(grp_2019_avec_nuance_max['Nuance'].isna().sum())

3642


In [6]:
grp_2019_avec_nuance.head(20)

NameError: name 'grp_2019_avec_nuance' is not defined

In [23]:
liste_candidats_nuance[(liste_candidats_nuance["Prénom"] == "Jean") & (liste_candidats_nuance["Nom"] == "Chabry")]

,annee,dep,Nom,Prénom,Nuance


In [24]:
grp_2013_avec_nuance.head()

,Région siège,Département siège,Arrondissement siège,Commune siège,N° SIREN,Nom du groupement,Nature juridique,Syndicat à la carte,Groupement interdépartemental,Date de création,...,Mode de financement,Civilité Président,Prénom Président,Nom Président,annee,dep_x,dep_y,Nom,Prénom,Nuance
0,82 - Rhône-Alpes,01 - Ain,2 - Bourg-en-Bresse,210100533 - Bourg-en-Bresse,240100628,CA Bourg en Bresse Agglomération,CA,0,0,28/11/2000,...,FPU,M.,Michel,FONTAINE,2008,01,ZD,FONTAINE,Michel,LMAJ
1,82 - Rhône-Alpes,01 - Ain,4 - Nantua,210101994 - Jujurieux,200029999,CC Rives de l'Ain - Pays du Cerdon,CC,0,0,25/11/2011,...,FPU,M,Jean,CHABRY,2008,01,NaN,NaN,NaN,NaN
2,82 - Rhône-Alpes,01 - Ain,2 - Bourg-en-Bresse,210100939 - Châtillon-sur-Chalaronne,200035210,CC Chalaronne Centre,CC,0,0,01/01/2013,...,ASZSE,M,Patrice,MORANDAS,2008,01,NaN,NaN,NaN,NaN
3,82 - Rhône-Alpes,01 - Ain,2 - Bourg-en-Bresse,210102661 - Montrevel-en-Bresse,240100156,CC de Montrevel - en - Bresse,CC,0,0,01/01/2002,...,ASZSE,M.,Jean Pierre,ROCHE,2008,01,NaN,NaN,NaN,NaN
4,82 - Rhône-Alpes,01 - Ain,4 - Nantua,210102836 - Oyonnax,240100172,CC d'Oyonnax,CC,0,0,19/06/2000,...,FPU,M.,Alexandre,TACHDJIAN,2008,01,NaN,NaN,NaN,NaN
